# The style and industry factor model

The platform has its own risk model, built in the warehouse every day. It is a
fundamental factor model in the style of Barra: on each session, one weighted
cross-sectional regression explains the returns of the universe with the industry of
each name and eight style characteristics. This notebook states the model, shows what it
explains, and tests whether its risk numbers are right.

The model is in three tables of the `factors` schema:

| Table | Grain | Holds |
|---|---|---|
| `style_exposures` | name × session | the industry and the eight style z-scores of D, and the regression weight |
| `style_factor_returns` | session | the market, the eight style returns and the 13 industry returns earned on D |
| `style_residuals` | name × session | the return split into market, industry, style and a stock-specific residual |

The other factor families in `factors.factor_returns` (quintile long-shorts, PCA
eigenportfolios, the published Fama-French factors and their mimics) are for
comparison. The style model is the one a portfolio optimiser reads.


In [ ]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import polars as pl

from sdp import dal

wh = dal.warehouse()   # A read-only connection to the dbt warehouse.
STYLES = ["size", "liquidity", "beta", "momentum", "reversal", "volatility",
          "dividend_yield", "high_52w"]

## 1. The model

For name $i$ with exposures known at the close of session $t$, the return of session
$t+1$ is

$$r_{i,t+1} = f^{mkt}_{t+1} + f^{ind}_{g(i),t+1} + \sum_{k=1}^{8} z_{ik,t}\, f^{k}_{t+1} + u_{i,t+1}$$

- $g(i)$ is the Fama-French 12 industry of the SIC code, plus `Unknown` for a name with
  no code yet.
- $z_{ik,t}$ is the z-score of style $k$ on session $t$: winsorised at 5 robust standard
  deviations, centred on the weighted mean, scaled by the standard deviation and clipped
  at ±3. A missing characteristic takes 0, the mean.
- The regression is weighted least squares with weight $\sqrt{\text{cap}}$. A large name
  counts more, but not in proportion to its size, so the mega-caps do not decide the
  fit alone.
- The industry returns are constrained so that their cap-weighted sum is 0. The market
  return is then the cap-weighted mean of the industry coefficients, and each industry
  return is the return of that industry over the market.
- Each style return is the return of a portfolio with unit exposure to that style and
  no exposure to the other styles or the industries: a pure factor.
- $u_{i,t+1}$ is the stock-specific return. `spec_vol` is the forecast of its volatility
  from the sessions before: an exponentially weighted volatility of the residuals of the
  name, shrunk toward its size decile and scaled by the recent cross-section of
  residuals. `resid_z` is $u$ in units of `spec_vol`.

The eight styles, all known at the close of $t$:

| Style | Characteristic |
|---|---|
| size | log market cap of the share class |
| liquidity | log of 20-day dollar volume over market cap |
| beta | 252-session beta to the cap-weighted return of the universe |
| momentum | return from 12 months to 1 month before |
| reversal | return over the last 5 sessions |
| volatility | standard deviation of the last 60 daily returns |
| dividend_yield | trailing 12-month cash dividends over price |
| high_52w | close over the highest close of 252 sessions |

Value and quality styles need the financial statements, which the platform does not
ingest yet.


In [ ]:
last = wh.sql("select max(date) from factors.style_exposures").fetchone()[0]
exp = wh.sql(f"""
    select ticker, industry, market_cap, weight,
           {", ".join(f"z_{s}" for s in STYLES)}
    from factors.style_exposures
    where date = date '{last}'
""").pl()
print(f"exposures on {last}: {len(exp):,} names")
exp.sort("market_cap", descending=True, nulls_last=True).head(10)

## 2. The exposures

The z-scores are centred on the weighted mean, so the weighted universe has an exposure
near 0 to each style. The correlation matrix below shows how
far the styles overlap. A high correlation between two styles makes their pure factor
returns noisy, because the regression must separate them.


In [ ]:
Z = exp.select([f"z_{s}" for s in STYLES]).to_numpy()
ok = np.isfinite(Z).all(1)
C = np.corrcoef(Z[ok], rowvar=False)
fig = px.imshow(C, x=STYLES, y=STYLES, zmin=-1, zmax=1, color_continuous_scale="RdBu_r",
                text_auto=".2f", title=f"Correlation of the style exposures on {last}")
fig.update_layout(height=520)
fig.show()

In [ ]:
ind = (exp.group_by("industry")
       .agg(names=pl.len(), cap_bn=(pl.col("market_cap").sum() / 1e9).round(0))
       .sort("cap_bn", descending=True))
ind

## 3. The factor returns

Each row of `style_factor_returns` is dated on the session that earns the return. The
plot compounds the market and the eight pure styles.


In [ ]:
fr = wh.sql("select * from factors.style_factor_returns order by date").pl()
cum = fr.select("date", *[pl.col(c).fill_null(0).add(1).log().cum_sum().exp().sub(1).alias(c)
                          for c in ["market", *STYLES]])
fig = px.line(cum.unpivot(index="date", variable_name="factor", value_name="cumulative"),
              x="date", y="cumulative", color="factor",
              title="Cumulative return of the market and the pure style factors")
fig.update_layout(height=460, yaxis_tickformat=".0%")
fig.show()

In [ ]:
stats = (fr.select(["market", *STYLES])
         .unpivot(variable_name="factor", value_name="ret").drop_nulls()
         .group_by("factor", maintain_order=True)
         .agg(ann_ret=(pl.col("ret").mean() * 252),
              ann_vol=(pl.col("ret").std() * np.sqrt(252)))
         .with_columns(sharpe=pl.col("ann_ret") / pl.col("ann_vol"),
                       t=pl.col("ann_ret") / pl.col("ann_vol") * np.sqrt(len(fr) / 252)))
stats.with_columns(pl.col("ann_ret", "ann_vol", "sharpe", "t").round(3))

A pure style return is a hedged bet on one characteristic. Its volatility is much lower
than that of the market, and a t near 2 over five years is weak evidence of a premium.
Most of these factors are for risk, not for return: they tell an optimiser which bets a
portfolio carries without meaning to.

The published Fama-French factors are sorts with NYSE breakpoints, rebalanced once a
year or once a month. The style factors are daily regressions. The table compares the
two where they measure the same idea, over the sessions where both exist.


In [ ]:
pairs = [("market", "mkt_rf"), ("size", "smb"), ("momentum", "mom")]
ff = wh.sql("""
    select date, factor, ret from factors.factor_returns
    where family = 'ff' and factor in ('mkt_rf', 'smb', 'mom')
""").pl().pivot(on="factor", index="date", values="ret")
both = fr.select("date", "market", "size", "momentum").join(ff, on="date", how="inner")
pl.DataFrame([{"style": a, "fama_french": b, "sessions": both.height,
               "correlation": round(float(np.corrcoef(both[a].fill_null(0),
                                                      both[b].fill_null(0))[0, 1]), 3)}
              for a, b in pairs])

The size style has the opposite sign to SMB by construction: SMB is small minus big,
and the size style pays a large exposure.

## 4. How much the model explains

`r2` is the weighted share of the cross-section of returns that the model explains on
one session. It is high on days when one theme moves the market, and low on quiet days
when stock-specific news dominates.


In [ ]:
r2 = fr.select("date", "r2", "n_names").with_columns(
    r2_63=pl.col("r2").rolling_mean(63))
fig = go.Figure()
fig.add_trace(go.Scatter(x=r2["date"], y=r2["r2"], mode="lines", name="r2, daily",
                         line=dict(color="#9aa0a6", width=0.7)))
fig.add_trace(go.Scatter(x=r2["date"], y=r2["r2_63"], mode="lines", name="63-session mean",
                         line=dict(color="#cf222e", width=2)))
fig.update_layout(title=f"Share of the cross-section explained (median "
                        f"{r2['r2'].median():.1%})", yaxis_tickformat=".0%", height=400)
fig.show()

## 5. Are the risk numbers right?

A risk model is right when a return divided by its predicted volatility has a standard
deviation of 1. This ratio is the bias statistic. Above 1, the model predicts too little
risk. Below 1, too much. With fat tails, the standard deviation is above 1 even for a
right model, so the table also gives a robust version: the median absolute value over
0.6745, which is 1 for a normal distribution.

**Specific risk.** `resid_z` is the residual over `spec_vol`, a forecast that reads only the
sessions before, with at least 21 residuals of the name. So each `resid_z` is out of sample.


In [ ]:
spec = wh.sql("""
    select date, resid_z
    from factors.style_residuals
    where resid_z is not null
""").pl()
year = spec.with_columns(year=pl.col("date").dt.year()).group_by("year").agg(
    n=pl.len(),
    bias=pl.col("resid_z").std(),
    robust_bias=(pl.col("resid_z").abs().median() / 0.6745),
    beyond_4=(pl.col("resid_z").abs() > 4).mean()).sort("year")
year.with_columns(pl.col("bias", "robust_bias").round(3), pl.col("beyond_4").round(4))

The two numbers point in opposite directions, and both are real. The bias is near 1:
the forecast has the right scale on average, and its regime factor keeps the scale when
the market calms or turns. The robust bias is about 0.7: on a typical day, a name moves
less than `spec_vol` says, because the forecast includes its tail days. A normal
distribution puts about 0.006% of the days beyond 4σ, and the residuals put far more
there (earnings, deals, news). One number cannot describe both. An optimiser
that reads `spec_vol` needs a limit on the weight of any one name.

**Factor risk.** Each factor return is divided by the volatility of that factor over the
63 sessions before it.


In [ ]:
rows = []
for f in ["market", *STYLES]:
    # Momentum needs a year of history, so its exposures are near 0 in the first year and
    # its early returns are tiny. Each test starts after the first 252 sessions.
    s = fr.select("date", pl.col(f).alias("ret")).drop_nulls().slice(252)
    s = s.with_columns(pred=pl.col("ret").rolling_std(63).shift(1)).drop_nulls()
    z = (s["ret"] / s["pred"]).to_numpy()
    rows.append({"factor": f, "sessions": z.size, "bias": round(float(z.std()), 3),
                 "robust_bias": round(float(np.median(np.abs(z)) / 0.6745), 3)})
pl.DataFrame(rows)

## 6. How much of a stock's risk is specific

For a typical name, most of the daily variance is stock-specific. The factor part is
what a portfolio of many names keeps, because the specific parts diversify away. The plot
shows the specific share of the variance of each name over the last year.


In [ ]:
share = wh.sql("""
    select ticker,
           var_samp(resid) / nullif(var_samp(ret), 0) as specific_share,
           any_value(weight_cap) as cap
    from factors.style_residuals
    where date > (select max(date) from factors.style_residuals) - interval 1 year
    group by ticker
    having count(*) >= 200
""").pl()
fig = px.histogram(share.filter(pl.col("specific_share") <= 1.5), x="specific_share",
                   nbins=80, title=f"Specific share of the daily variance, {len(share):,} "
                                   f"names (median {share['specific_share'].median():.0%})")
fig.update_layout(height=380, xaxis_tickformat=".0%")
fig.show()

## Takeaways

- The model explains a median of about a sixth of the daily cross-section, which is
  usual for a daily model with no value or quality styles. It explains more on the days
  when one theme drives the market.
- The pure style factors have low volatility and weak premia. They are risk factors:
  they show the bets that a portfolio carries.
- The factor volatilities of the last 63 sessions are about right: the bias is near 1.
- The specific volatility has the right scale on average and follows the regime, but
  it is too high for a typical day and too low for the tails. An
  optimiser that reads it needs a limit on the weight of one name.
- Next: a factor covariance on the style factor returns, so that the model gives the
  risk of a whole portfolio. `risk_factor_models.ipynb` tests that covariance against the
  statistical models.